# Case Linking — Reference Data Tests

Verifies the static lookup tables that drive everything else:
- bronze_case_link_reasons_aria (26 rows, IDs 2–27)
- bronze_case_link_reasons_ccd (17 rows, CLRC001–CLRC017)
- silver_case_link_reason_mappings


In [0]:
########################
# test Setup
########################

state_under_test = "caseLinking"
run_tag_default = "reference_data"

dbutils.widgets.text("run_tag", run_tag_default)
run_tag = dbutils.widgets.get("run_tag") or run_tag_default

In [0]:
import os
import sys
import time as perf_time
from datetime import datetime

from pyspark.sql.functions import col
import inspect

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/Case_Linking_Tests"
os.makedirs(test_results_path, exist_ok=True)

for _mod in list(sys.modules.keys()):
    if _mod.startswith("Test_Functions") or _mod == "models.test_result":
        del sys.modules[_mod]

from models.test_result import TestResult
from Test_Functions.report_helpers import (
    build_report_folder,
    write_csv, write_xlsx, write_html,
    count_by_status,
    create_run, create_results,
)

print(f"User: {run_user}")
print(f"state_under_test: {state_under_test}")
print(f"run_tag: {run_tag}")
print(f"Results folder root: {test_results_path}")

## Load reference tables

In [0]:
bronze_aria_reasons    = spark.table("hive_metastore.ariadm_active_appeals.bronze_case_link_reasons_aria")
bronze_ccd_reasons     = spark.table("hive_metastore.ariadm_active_appeals.bronze_case_link_reasons_ccd")
silver_reason_mappings = spark.table("hive_metastore.ariadm_active_appeals.silver_case_link_reason_mappings")

print(f"bronze_aria_reasons rows:    {bronze_aria_reasons.count()}")
print(f"bronze_ccd_reasons rows:     {bronze_ccd_reasons.count()}")
print(f"silver_reason_mappings rows: {silver_reason_mappings.count()}")

## Tests

Add a new cell per leg below. Each cell appends to `all_test_results`.

In [0]:
run_start_datetime = datetime.utcnow()
all_test_results = []
t0 = perf_time.time()

# ----------------------------------------------------------------
# Tests go here. Pattern:
#
#     def test_xxx(df):
#         try:
#             # checks...
#             return TestResult("<field>", "PASS", "<msg>", state_under_test, inspect.stack()[0].function)
#         except Exception as e:
#             return TestResult("<field>", "FAIL", f"Test crashed: {e}", state_under_test, inspect.stack()[0].function)
#
#     all_test_results.append(test_xxx(some_df))
# ----------------------------------------------------------------

# (no tests yet — added one leg at a time)

elapsed = perf_time.time() - t0
counts = count_by_status(all_test_results)

print(f"Tests complete in {elapsed:.1f}s")
print(f"  Total : {counts['TOTAL']}")
print(f"  Pass  : {counts['PASS']}")
print(f"  Fail  : {counts['FAIL']}")
print(f"  NoData: {counts['NO_DATA']}")
print(f"  Error : {counts['ERROR']}")

## Write reports + audit

In [0]:
folder, timestamp, _ = build_report_folder(test_results_path, f"{state_under_test}_{run_tag}")
print(f"Reports folder: {folder}")

try:
    csv_path  = write_csv(all_test_results, state_under_test, folder, timestamp)
    print(f"CSV  : {csv_path}")
except Exception as e:
    print(f"write_csv FAILED: {e}")

try:
    xlsx_path = write_xlsx(all_test_results, state_under_test, folder, timestamp)
    print(f"XLSX : {xlsx_path}")
except Exception as e:
    print(f"write_xlsx FAILED: {e}")

try:
    html_path = write_html(all_test_results, state_under_test, folder, timestamp, counts=counts)
    print(f"HTML : {html_path}")
except Exception as e:
    print(f"write_html FAILED: {e}")

try:
    run_id = create_run(
        spark,
        run_user=run_user,
        run_start_datetime=run_start_datetime,
        run_by_automation_name="Case_Linking_Tests",
        run_tag=run_tag,
        run_status="COMPLETED",
        state_under_test=state_under_test,
    )
    print(f"Created run_id={run_id}")
    n = create_results(spark, run_id, all_test_results)
    print(f"Wrote {n} result rows")
except Exception as e:
    print(f"audit write FAILED: {e}")